In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [5]:
contract = pd.read_csv(r'C:\Users\User-L\Desktop\Cientifico de datos-preparacion\Sprint 19 Final-Proyect/Datasets/contract.csv')
personal = pd.read_csv(r'C:\Users\User-L\Desktop\Cientifico de datos-preparacion\Sprint 19 Final-Proyect/Datasets/personal.csv')
internet = pd.read_csv(r'C:\Users\User-L\Desktop\Cientifico de datos-preparacion\Sprint 19 Final-Proyect/Datasets/internet.csv')
phone = pd.read_csv(r'C:\Users\User-L\Desktop\Cientifico de datos-preparacion\Sprint 19 Final-Proyect/Datasets/phone.csv')

In [6]:
merged = contract.merge(personal, on='customerID', how='left') \
             .merge(internet, on='customerID', how='left') \
             .merge(phone, on='customerID', how='left')

In [7]:
merged['churn'] = merged['EndDate'].apply(lambda x: 0 if x == 'No' else 1)

In [8]:
merged['TotalCharges'] = pd.to_numeric(merged['TotalCharges'], errors='coerce')

## PREPROCESAMIENTO Y FEATURE ENGENIERING

In [9]:
merged['BeginDate'] = pd.to_datetime(merged['BeginDate'], errors='coerce')
merged['EndDate'] = pd.to_datetime(merged['EndDate'], errors='coerce')

merged['tenure_days'] = (merged['EndDate'] - merged['BeginDate']).dt.days

reference_date = pd.to_datetime('2020-02-01')
merged['tenure_days'] = merged['tenure_days'].fillna(
    (reference_date - merged['BeginDate']).dt.days
)

merged['TotalCharges'] = pd.to_numeric(merged['TotalCharges'], errors='coerce')

merged = merged.fillna(0)

y = merged['churn']
X = merged.drop(['customerID', 'churn', 'EndDate', 'BeginDate'], axis=1)
X = pd.get_dummies(X, drop_first=True)

C:\Users\User-L\AppData\Local\Temp\ipykernel_8044\769627676.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  merged['EndDate'] = pd.to_datetime(merged['EndDate'], errors='coerce')


## ENTRENAMIENTO DE MODELOS

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Modelos

models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "LightGBM": LGBMClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results[name] = {
        "model": model,
        "pred": y_pred
    }

c:\Users\User-L\Documents\miniconda\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000744 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 705
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265353 -> initscore=-1.018328
[LightGBM] [Info] Start training from score -1.018328


## EVALUACION Y SELECCION

In [11]:
evaluation = []
for name, res in results.items():
    y_pred = res["pred"]
    model = res["model"]

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:,1]
    else:
        y_proba = None

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    roc = roc_auc_score(y_test, y_proba) if y_proba is not None else None

    evaluation.append([name, acc, f1, roc])

# Resultados
results_df = pd.DataFrame(evaluation, columns=["Model", "Accuracy", "F1", "ROC-AUC"])
results_df.sort_values(by="F1", ascending=False)

,Model,Accuracy,F1,ROC-AUC
3,LightGBM,0.852378,0.680982,0.901542
1,Logistic,0.803407,0.601439,0.843432
2,RandomForest,0.805536,0.586103,0.840285
0,Dummy,0.734564,0.000000,0.500000


## CONCLUSIONES

Conclusión del modelado de churn – Interconnect

Tras el desarrollo y evaluación de múltiples modelos de clasificación, se identificó que LightGBM es el modelo con mejor desempeño general para la predicción de cancelación de clientes (churn).

Resultados clave
LightGBM alcanzó:
F1-score: 0.681
ROC-AUC: 0.902
Mejor equilibrio entre detección de clientes en riesgo y precisión
Modelos comparativos:
Random Forest y Regresión Logística mostraron desempeño moderado
DummyClassifier confirmó el baseline (sin capacidad predictiva real)

El modelo LightGBM:

Identifica eficazmente a clientes con alta probabilidad de cancelar
Reduce significativamente el riesgo de perder clientes sin intervención
Permite priorizar acciones de retención (promociones, planes especiales)

El alto ROC-AUC (0.90) indica que el modelo discrimina muy bien entre clientes que cancelarán y los que no.

El F1-score (~0.68) refleja un balance sólido entre:

Detectar clientes en riesgo (recall)
Evitar falsas alarmas (precision)

Trade-offs observados
Modelos más simples (Logistic Regression):
Más interpretables
Menor rendimiento
Modelos avanzados (LightGBM):
Mayor precisión predictiva
Menor interpretabilidad directa

Se priorizó rendimiento debido al impacto directo en retención de clientes.

Recomendación:

Se recomienda implementar LightGBM como modelo principal de predicción de churn, debido a:

Su superior desempeño en todas las métricas clave
Su capacidad para capturar relaciones complejas en los datos
Su potencial impacto en la reducción de cancelaciones

Próximos pasos
Implementar el modelo en un entorno productivo (batch o near real-time)
Integrar el sistema con el equipo de marketing para acciones de retención
Analizar la importancia de variables para entender los drivers de churn
Realizar tuning adicional para maximizar el F1-score
Monitorear el desempeño del modelo en producción